In [ ]:
# NB : This reload your library after each edit, 
# so you don't have to restart the kernel
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# ML
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Loading Data

In [ ]:
df_diamonds = sns.load_dataset('diamonds')

In [ ]:
df_diamonds.head()

In [ ]:
df_diamonds.info()

In [ ]:
df_diamonds.describe()

In [ ]:
df_diamonds.isna().sum()

## Exploratory Data Analysis

In [ ]:
sns.histplot(data=df_diamonds,
             x="price")

In [ ]:
px.scatter_3d(data_frame=df_diamonds,
              x="x",y="y",z="z", color="price")

In [ ]:
sns.pairplot(data=df_diamonds);

In [ ]:
sns.heatmap(df_diamonds.select_dtypes(include="number").corr(),
           annot=True,
           cmap="coolwarm")

In [ ]:
df_diamonds.select_dtypes(include="category")\
            ["color"].value_counts()

In [ ]:
import scipy.stats as stats
groups = [group["price"].values for name, group in df_diamonds.groupby("color", observed=False)]


print("--- Test ANOVA à un facteur ---")
f_stat, p_value_anova = stats.f_oneway(*groups)
print(f"Statistique F : {f_stat:.2f}")
print(f"P-value       : {p_value_anova:.2e}")

if p_value_anova < 0.05:
    print("Conclusion : On rejette H0. La couleur a un impact significatif sur le prix moyen.\n")
else:
    print("Conclusion : On ne peut pas rejeter H0. La couleur ne semble pas impacter le prix.\n")

## Data Cleaning

In [ ]:
def keep_not_null(row) :
    if 0 in row.values : return False
    return True

# Pour le test sur une ligne :
keep_not_null(df_diamonds.loc[11182,:])
    

In [ ]:
df_clean = df_diamonds[df_diamonds.apply(keep_not_null,axis=1)]

In [ ]:
df_clean[df_clean["x"] == 0]

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["price"])
y = df_clean["price"]

X_train, X_test, y_train, y_test  = train_test_split(X,y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Data Preprocessing
### Categorical Data

In [ ]:
df_cat = df_clean.select_dtypes(include="category")
df_cat.describe()

In [ ]:
cat_pipe = Pipeline(
    [ ("cat_imp",SimpleImputer(strategy="most_frequent"))
      ,("ohe",OneHotEncoder(drop="first",sparse_output=False))
        ])
cat_pipe

### Numerical Data

In [ ]:
num_pipe = Pipeline(
    [("knn_imp", KNNImputer(n_neighbors=5))
     ,("scaler", StandardScaler())
      ])
num_pipe
    

## Final Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [("numeric",num_pipe, make_column_selector(dtype_include="number"))
    ,("categorical", cat_pipe, make_column_selector(dtype_exclude="number"))
      ]).set_output(transform="pandas")
preprocessor

In [ ]:

# Normalement j'ai pas besoin de preproc mais pour vérfier 
preprocessor.fit(X_train)
X_train_scaled = preprocessor.transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

## Training 
### Trying a few models

In [ ]:
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

df_clean["price"].max()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true,y_pred) -> dict[float] : 
    mae = mean_absolute_error(y_true,y_pred)
    mse = mean_squared_error(y_true,y_pred)
    r2  = r2_score(y_true,y_pred) 
    scores = {"mae":mae,"mse":mse,"r2":r2}
    print(scores)
    return scores
    

In [ ]:
lin = LinearRegression()
lin.fit(X_train_scaled,y_train) 
y_pred = lin.predict(X_test_scaled)
evaluate_model(y_test,y_pred)

In [ ]:
forest = RandomForestRegressor()
forest.fit(X_train_scaled,y_train)
y_pred = forest.predict(X_test_scaled)
evaluate_model(y_test,y_pred)

In [ ]:
knn = KNeighborsRegressor()
knn.fit(X_train_scaled,y_train)
y_pred = knn.predict(X_test_scaled)
evaluate_model(y_test,y_pred)

In [ ]:
svm = SVR()
svm.fit(X_train_scaled,y_train)
y_pred = svm.predict(X_test_scaled)
evaluate_model(y_test,y_pred)

In [ ]:
! pip freeze | grep tensor

In [ ]:
# Bonus : Deep Learning model with Tensorflow

## Optimizing Hyperparameters

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {"n_neighbors": range(2,30)}


search = GridSearchCV(KNeighborsRegressor(), grid, verbose=1)

search.fit(X_train_scaled,y_train)

In [ ]:
y_pred = search.predict(X_test_scaled)
print(search.best_params_)
evaluate_model(y_test,y_pred)

In [ ]:
grid = {"n_estimators": [100,150,300,500]
        #,"max_depth" : [None, 2,5,10]
        #,"min_samples_leaf" : [1,5,10,50]
       }

search = GridSearchCV(RandomForestRegressor(),grid,verbose=1, cv=3)
search.fit(X_train_scaled,y_train)


In [ ]:
y_pred = search.predict(X_test_scaled)
print(search.best_params_)
evaluate_model(y_test,y_pred)

### Final Train

In [ ]:
forest = RandomForestRegressor(**search.best_params_)
forest.fit(X_train_scaled,y_train)
y_pred = forest.predict(X_test_scaled)
evaluate_model(y_test,y_pred)

## Saving Model

In [ ]:
import pickle 
import os 

In [ ]:
! ls

In [ ]:
# Saving the preprocessor 
model_path = "models"
if not os.path.exists(model_path) : 
    os.mkdir(model_path)
with open(os.path.join(model_path,"preproc.pkl"),"wb") as f:
    pickle.dump(preprocessor,f)

In [ ]:
# Saving the model 
with open(os.path.join(model_path,"model.pkl"),"wb")  as f:
    pickle.dump(forest,f)

In [ ]:
! tree

## NB : Loading model

In [ ]:
with open(os.path.join(model_path,"preproc.pkl"),"rb")  as f:
    new_preproc = pickle.load(f)
new_preproc